# Detect unique tokens in the dataset
The purpose of this notebook is to identify some sort of distribution of the tokens, and hopefully detect some tokens that rarely occur in sentences

To download the data use the following command
```
python -c '
from datasets import load_dataset
import os
from datasets import Dataset

os.makedirs("data", exist_ok=True)

# Streaming=True avoids the 300GB download
dataset = load_dataset("allenai/c4", "en", split="train", streaming=True)
small_subset = dataset.take(1000)

# Save to your local project folder
Dataset.from_list(list(small_subset)).to_json("data/c4_sample.jsonl")

print("Saved 1000 rows to data/c4_sample.jsonl")
'
```

In [50]:
# Load data from data folder

import json
import os
# Define the path clearly
base_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
file_path = os.path.join(base_dir, 'data', 'c4_sample.jsonl')

with open(file_path, 'r') as json_file:
    json_list = list(json_file)


In [51]:

forbidden_chars = """
!@#$%,.:?;()_"®“0123456789
"""
# Move this OUTSIDE the loop so it's only created once
translation_table = str.maketrans('', '', forbidden_chars)

cleaned_tokens = []

for json_str in json_list:
    # 1. Load the dictionary
    data = json.loads(json_str)
    raw_text = data["text"]
    
    # 2. Clean the entire string at once (Maximum Speed)
    clean_text = raw_text.translate(translation_table)
    
    # 3. Split into tokens now that it's clean
    tokens = clean_text.split()
    
    cleaned_tokens.append(tokens)



In [52]:
# Count each of the tokens
dictionary: dict[str,int] = {}
for token_list in cleaned_tokens:
    for token in token_list:
        lowered_token = token.lower()
        if lowered_token in dictionary:
            dictionary[lowered_token] += 1
        else:
            dictionary[lowered_token] = 1


In [53]:
dictionary

{'beginners': 2,
 'bbq': 9,
 'class': 46,
 'taking': 69,
 'place': 198,
 'in': 6309,
 'missoulado': 1,
 'you': 3790,
 'want': 326,
 'to': 10909,
 'get': 560,
 'better': 183,
 'at': 1692,
 'making': 145,
 'delicious': 18,
 'will': 1552,
 'have': 1747,
 'the': 18298,
 'opportunity': 77,
 'put': 134,
 'this': 1970,
 'on': 2690,
 'your': 2157,
 'calendar': 12,
 'now': 360,
 'thursday': 28,
 'september': 24,
 'nd': 14,
 'join': 46,
 'world': 232,
 'champion': 13,
 'tony': 7,
 'balay': 1,
 'from': 1614,
 'lonestar': 1,
 'smoke': 5,
 'rangers': 5,
 'he': 688,
 'be': 2275,
 'teaching': 29,
 'a': 8289,
 'beginner': 1,
 'level': 130,
 'for': 4215,
 'everyone': 105,
 'who': 588,
 'wants': 62,
 'with': 3096,
 'their': 951,
 'culinary': 4,
 'skillshe': 1,
 'teach': 22,
 'everything': 94,
 'need': 384,
 'know': 324,
 'compete': 8,
 'kcbs': 1,
 'competition': 37,
 'including': 181,
 'techniques': 24,
 'recipes': 17,
 'timelines': 3,
 'meat': 21,
 'selection': 35,
 'and': 10896,
 'trimming': 2,
 'plus

In [54]:
# Get the vocabulary
base_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
file_path = os.path.join(base_dir, 'notebooks', 'vocabulary', 'gpt2_tokenizer', 'tokenizer.json')

with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Navigate to the nested "vocab" key
vocabulary = data.get("model", {}).get("vocab", {})
vocabulary_set = {k.lower().strip().replace('ġ', ''): v for k, v in vocabulary.items()}
vocabulary_set

{'!': 5145,
 '"': 366,
 '#': 1303,
 '$': 720,
 '%': 4064,
 '&': 1222,
 "'": 705,
 '(': 357,
 ')': 1267,
 '*': 1635,
 '+': 1343,
 ',': 837,
 '-': 532,
 '.': 764,
 '/': 1220,
 '0': 657,
 '1': 352,
 '2': 362,
 '3': 513,
 '4': 604,
 '5': 642,
 '6': 718,
 '7': 767,
 '8': 807,
 '9': 860,
 ':': 1058,
 ';': 2162,
 '<': 1279,
 '=': 796,
 '>': 1875,
 '?': 5633,
 '@': 2488,
 'a': 317,
 'b': 347,
 'c': 327,
 'd': 360,
 'e': 412,
 'f': 376,
 'g': 402,
 'h': 367,
 'i': 1312,
 'j': 474,
 'k': 509,
 'l': 406,
 'm': 337,
 'n': 399,
 'o': 440,
 'p': 350,
 'q': 10662,
 'r': 374,
 's': 311,
 't': 309,
 'u': 471,
 'v': 569,
 'w': 370,
 'x': 2124,
 'y': 575,
 'z': 1976,
 '[': 685,
 '\\': 3467,
 ']': 2361,
 '^': 10563,
 '_': 4808,
 '`': 4600,
 '{': 1391,
 '|': 930,
 '}': 1782,
 '~': 5299,
 '¡': 94,
 '¢': 95,
 '£': 96,
 '¤': 97,
 '¥': 98,
 '¦': 99,
 '§': 100,
 '¨': 101,
 '©': 102,
 'ª': 103,
 '«': 104,
 '¬': 105,
 '®': 106,
 '¯': 107,
 '°': 108,
 '±': 109,
 '²': 110,
 '³': 111,
 '´': 112,
 'µ': 113,
 '¶': 114

In [56]:
import nltk

# 'words' is a general list of English words
# 'wordnet' is the high-precision lexical database
nltk.download('words')
nltk.download('wordnet')
nltk.download('omw-1.4') 

[nltk_data] Downloading package words to
[nltk_data]     /Users/emilhejlesen/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/emilhejlesen/nltk_data...
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/emilhejlesen/nltk_data...


True

In [ ]:
from nltk.corpus import words, wordnet

# 1. Create a master set of valid words

# We combine the general words list and WordNet for maximum coverage
english_vocab = set(w.lower() for w in words.words())
wordnet_vocab = set(w.lower() for w in wordnet.all_lemma_names())

print("Length of dictionary (before):", len(dictionary))
keys = set(dictionary.keys())
for key in keys:
    if key not in english_vocab and key not in wordnet_vocab:
        print("Deleted key:", key)
        del dictionary[key]
print("Length of dictionary (after):", len(dictionary))

Length of dictionary (before): 35776
Deleted key: adaptations
Deleted key: escaping
Deleted key: changehaleakala
Deleted key: carecompassion
Deleted key: verbessern
Deleted key: insteadbenefits
Deleted key: tiekie
Deleted key: ohms
Deleted key: «
Deleted key: rulesbut
Deleted key: 'bold
Deleted key: usedif
Deleted key: opportunityin
Deleted key: katherine
Deleted key: sprays
Deleted key: searchprepared
Deleted key: widerelevance
Deleted key: toxins
Deleted key: once-a-year
Deleted key: availableideal
Deleted key: itone
Deleted key: ventured
Deleted key: pmdate
Deleted key: ander
Deleted key: in-lawsand
Deleted key: ieagle
Deleted key: possiblethis
Deleted key: dmc
Deleted key: dragonafter
Deleted key: strips
Deleted key: mealswhile
Deleted key: perfects
Deleted key: 'lucky
Deleted key: bucks
Deleted key: arrived
Deleted key: diggingsharpness
Deleted key: avoids
Deleted key: preferenceno
Deleted key: replaced
Deleted key: society”
Deleted key: modules
Deleted key: rhoton
Deleted key: dj

In [58]:
dictionary["escaping"]

KeyError: 'escaping'